# AI-RAG · Week 11 — Notebook 2
## Eval as Code & the 8-KPI Dashboard

This notebook assembles the **8-KPI dashboard** from scored rows, persists per-query cost/latency to `results.db` tagged by `kb_version`, renders the baseline report, and sets **thresholds from a baseline**. Runs **offline** end-to-end. It reuses the metric machinery from Notebook 1 — included below under *Prerequisites* so this notebook runs on its own — then spends its narration on what's new in Day 2.

> Code is verbatim from `src/eval/*.py` and `scripts/*.py`; only notebook glue (paths, inlined imports, and replacing the `argparse` CLI wrapper with direct cells) differs.

### Setup
This section prepares the notebook environment before the evaluation begins.

It performs three setup tasks:

1. Checks whether `pydantic` is installed and installs it if necessary. Pydantic is used to validate the structure of the golden-set records.
2. Locates the existing CBSE Class 9 golden set at `golden_set/capstone_v1.jsonl` and checks that it contains 20 entries.
3. Sets `EVAL_OFFLINE=1` to use the deterministic offline evaluation path without requiring an OpenAI API key or incurring API costs.

**Important:** The original notebook contains embedded HR-policy questions and writes them to the golden-set file. We retain the CBSE golden set prepared in Day 1 instead. The original source files remain unchanged.

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------

# 1) Ensure pydantic is available
# 2) Use the existing CBSE golden set
# 3) Force the OFFLINE proxy path

try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pydantic"]
    )

import os
from pathlib import Path

# Force offline evaluation
os.environ["EVAL_OFFLINE"] = "1"

# Locate the existing CBSE golden set
golden_path = Path("golden_set/capstone_v1.jsonl")

if not golden_path.exists():
    raise FileNotFoundError(
        f"CBSE golden set not found at: {golden_path}"
    )

# Count active records (ignore blank lines and // comments)
entry_count = sum(
    1 for line in golden_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("//")
)

print(
    "Using existing golden set:",
    entry_count,
    "entries · EVAL_OFFLINE =",
    os.environ["EVAL_OFFLINE"]
)

Using existing golden set: 20 entries · EVAL_OFFLINE = 1


In [2]:

from pathlib import Path
import json

golden_path = Path("golden_set/capstone_v1.jsonl")

with golden_path.open(encoding="utf-8") as f:
    first_entry = json.loads(next(line for line in f if line.strip()))

print("First ID:", first_entry["id"])
print("First question:", first_entry["question"])


First ID: g001
First question: What do R1, R2, and R3 mean in the revised language scheme?


### Prerequisites from Notebook 1

You met these in Notebook 1 — the golden loader, the four metric functions, and the row-builder. They're re-established here so this notebook is self-contained; the new material starts at section 1.

The prerequisite code re-establishes three components from Notebook 1:

1. **Golden-set loader (`golden.py`)** – reads and validates the 20 CBSE Class 9 questions and their reference answers. These reference answers act as the ground truth against which the evaluation is performed.

2. **RAGAS metrics (`ragas_metrics.py`)** – defines the four evaluation metrics: faithfulness, answer relevance, context precision and context recall. Since this notebook runs in offline mode, it uses deterministic proxy calculations rather than live RAGAS evaluation.

3. **Evaluation harness (`harness.py`)** – builds the evaluation rows containing each question, its simulated answer, retrieved context, reference answer, cache status, latency and cost.

**CBSE adaptation:** The original notebook's evaluation harness contains simulated failure cases designed around HR-policy questions. Since we are using the CBSE Class 9 golden set, those HR-specific cases need to be adapted. The harness will instead use each CBSE reference answer to create two relevant context chunks and add one unrelated distractor. This allows us to practise the evaluation and KPI workflow using our selected CBSE questions.

**Important:** These are synthetic evaluation rows, not answers generated by our actual RAG application. The offline metric values therefore demonstrate how the evaluation process works; they should not be interpreted as actual performance measurements of our RAG application.

All three prerequisite components are included below so that Notebook 2 can run independently of Notebook 1.

In [3]:
"""src/eval/golden.py — loader and validator for the capstone golden set.

Lineage: carried forward unchanged from W5. The golden set is the ground truth
that every later week's KPI is computed against — treat it as infrastructure,
not a one-off fixture. As of W10 it lives at `golden_set/capstone_v1.jsonl`
(the W5 set plus W7/W9 additions), 20 entries, ids g001..g020.

A golden entry has four fields:
  • id            — stable identifier (g001..g020). Referenced in eval runs
                    and across docs.
  • question      — exactly as a real user would phrase it.
  • ideal_answer  — full reference text. RAGAS uses this as `ground_truth`;
                    the judge uses it as the ideal answer. For open-ended
                    questions this may be a properties dict.
  • notes         — category (happy/harder/edge), source doc, why it matters.
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field, field_validator

_ID_PATTERN = re.compile(r"^g\d{3}$")


class GoldenEntry(BaseModel):
    """One row of the golden set."""

    id: str = Field(..., description="Stable identifier g001..g999")
    question: str = Field(..., min_length=1)
    ideal_answer: str | dict[str, Any] = Field(...)
    notes: str = Field(default="", description="Category, source, why this matters")

    @field_validator("id")
    @classmethod
    def _id_shape(cls, v: str) -> str:
        if not _ID_PATTERN.match(v):
            raise ValueError(f"id must match g001..g999; got {v!r}")
        return v

    @property
    def ground_truth(self) -> str:
        """RAGAS expects a string ground_truth. Flatten dict ideals to text."""
        if isinstance(self.ideal_answer, dict):
            return json.dumps(self.ideal_answer)
        return self.ideal_answer


def load_golden_set(path: str | Path) -> list[GoldenEntry]:
    """Read a .jsonl file. Each non-blank, non-comment line is one entry."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Golden set not found at {p}")
    entries: list[GoldenEntry] = []
    with p.open() as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line or line.startswith("//"):
                continue
            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"{p}:{lineno} — malformed JSON: {e.msg}") from e
            entries.append(GoldenEntry.model_validate(data))
    return entries


def validate_golden_set(path: str | Path) -> dict[str, Any]:
    """Validate the file. Returns a summary dict; raises on errors."""
    entries = load_golden_set(path)

    ids = [e.id for e in entries]
    if len(ids) != len(set(ids)):
        seen: set[str] = set()
        dupes = [i for i in ids if i in seen or seen.add(i)]  # noqa: B015
        raise ValueError(f"Duplicate ids in {path}: {sorted(set(dupes))}")

    def _bucket(note: str) -> str:
        low = note.lower()
        if low.startswith("happy"):
            return "happy"
        if low.startswith("harder") or low.startswith("multi"):
            return "harder"
        if low.startswith("edge"):
            return "edge"
        return "uncategorised"

    counts: dict[str, int] = {}
    for e in entries:
        counts[_bucket(e.notes)] = counts.get(_bucket(e.notes), 0) + 1

    return {"path": str(path), "n": len(entries), "ids": ids, "coverage": counts}


### Golden-set Loader and Validator

This section defines the structure and validation rules for our golden set.

The `GoldenEntry` Pydantic model specifies the four fields required for each question: `id`, `question`, `ideal_answer` and `notes`. It also validates that every question ID follows the expected format (`g001` to `g999`).

The `load_golden_set()` function reads the JSONL file, converts each record into a `GoldenEntry` object and checks that the record follows the defined structure.

The `ground_truth` property ensures that every ideal answer can be provided as a string for RAGAS evaluation, converting dictionary-based answers into JSON text when necessary.

The `validate_golden_set()` function performs additional checks for duplicate IDs and counts the questions under the categories happy, harder and edge.

**In our project:** This loader is reused with the CBSE Class 9 golden set containing 20 questions. The same validation logic works without modification because the CBSE records follow the required four-field structure.

In [4]:

"""src/eval/ragas_metrics.py — the four RAG-specific metrics, two ways to run.

W11 standardises on RAGAS for the four RAG metrics:
  • faithfulness        — is the answer supported by the retrieved context?
  • answer_relevance    — does the answer address the question?
  • context_precision   — are the top-K retrieved chunks the right ones?
  • context_recall      — did we retrieve everything we needed?

Each public function takes a list of `rows` and returns a `MetricResult`
(per-row scores + an aggregate). A row is a dict with:
    {"id", "question", "answer", "contexts": [str, ...], "ground_truth"}

TWO MODES
---------
LIVE  (default when an API key is set and `ragas` is importable):
      real RAGAS metrics, each backed by an LLM judge call per row. This is
      what produces the numbers you put in the snapshot and defend at M2.

OFFLINE (set EVAL_OFFLINE=1, or when `ragas` isn't installed):
      deterministic token-overlap proxies. These are NOT RAGAS numbers.
      They exist so the evaluation suite runs without an API key.
      Every offline MetricResult is labelled "offline-proxy".

This split allows the evaluation suite to run offline and produce real
numbers later when live evaluation is enabled.
"""
from __future__ import annotations

import os
import re
from dataclasses import dataclass, field


# ─── Result container ────────────────────────────────────────────────────────
@dataclass
class MetricResult:
    """One metric's outcome across all rows."""

    name: str
    mode: str
    aggregate: float
    per_row: dict[str, float] = field(default_factory=dict)

    def __str__(self) -> str:
        return f"{self.name}={self.aggregate:.3f} ({self.mode}, n={len(self.per_row)})"


# ─── Mode detection ──────────────────────────────────────────────────────────
def offline_mode() -> bool:
    """Offline if explicitly forced, or if ragas / an API key is unavailable."""
    if os.getenv("EVAL_OFFLINE", "").strip() in {"1", "true", "TRUE", "yes"}:
        return True
    if not os.getenv("OPENAI_API_KEY"):
        return True
    try:
        import ragas  # noqa: F401
    except Exception:
        return True
    return False


# ─── Token helpers (offline proxy) ───────────────────────────────────────────
_STOP = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on", "for",
    "with", "is", "are", "was", "were", "be", "been", "as", "at", "by", "it",
    "this", "that", "these", "those", "from", "your", "you", "may", "must",
    "can", "do", "does", "how", "what", "who", "i", "my", "me", "we", "our",
    "per", "up", "any", "all", "via", "not", "no", "they", "their",
}


def _tokens(text: str) -> set[str]:
    if not text:
        return set()
    raw = re.findall(r"[a-z0-9]+", text.lower())

    # CBSE adaptation: retain short alphanumeric codes such as R1, R2 and R3.
    return {
        t for t in raw
        if (len(t) > 2 or any(ch.isdigit() for ch in t)) and t not in _STOP
    }


def _overlap_fraction(a: set[str], b: set[str]) -> float:
    """Fraction of a's tokens that appear in b. 0..1. Empty a -> 0.0."""
    if not a:
        return 0.0
    return len(a & b) / len(a)


# ─── Offline proxies (deterministic, dependency-free) ────────────────────────
def _proxy(name: str, rows: list[dict], fn) -> MetricResult:
    per_row = {r["id"]: round(fn(r), 4) for r in rows}
    agg = round(sum(per_row.values()) / len(per_row), 4) if per_row else 0.0
    return MetricResult(
        name=name,
        mode="offline-proxy",
        aggregate=agg,
        per_row=per_row
    )


def _faithfulness_proxy(r: dict) -> float:
    # How much of the answer is covered by the retrieved context?
    ans = _tokens(r.get("answer", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(ans, ctx)


def _answer_relevance_proxy(r: dict) -> float:
    # Does the answer address the question?
    q = _tokens(r.get("question", ""))
    ans = _tokens(r.get("answer", ""))
    if not q:
        return 0.0
    coverage = _overlap_fraction(q, ans)
    touches = 1.0 if (q & ans) else 0.0
    return round(0.5 * coverage + 0.5 * touches, 4)


def _context_precision_proxy(r: dict) -> float:
    # Of the retrieved chunks, what fraction are relevant to the ground truth?
    gt = _tokens(r.get("ground_truth", ""))
    contexts = r.get("contexts", []) or []
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if _tokens(c) & gt)
    return relevant / len(contexts)


def _context_recall_proxy(r: dict) -> float:
    # How much of the ground truth is present across the retrieved chunks?
    gt = _tokens(r.get("ground_truth", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(gt, ctx)


# ─── Live RAGAS ──────────────────────────────────────────────────────────────
def _ragas_score(name: str, rows: list[dict], metric, column: str) -> MetricResult:
    """Run one RAGAS metric over the rows. Imported lazily (heavy deps)."""
    from datasets import Dataset
    from ragas import evaluate

    ds = Dataset.from_dict({
        "question":     [r["question"] for r in rows],
        "answer":       [r["answer"] for r in rows],
        "contexts":     [list(r.get("contexts", [])) for r in rows],
        "ground_truth": [r.get("ground_truth", "") for r in rows],
    })
    result = evaluate(ds, metrics=[metric])
    df = result.to_pandas()

    per_row = {
        rows[i]["id"]: round(float(df[column].iloc[i]), 4)
        for i in range(len(rows))
    }
    agg = round(float(df[column].mean()), 4)
    return MetricResult(
        name=name,
        mode="live",
        aggregate=agg,
        per_row=per_row
    )


# ─── Public API — one function per metric ────────────────────────────────────
def score_faithfulness(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("faithfulness", rows, _faithfulness_proxy)
    from ragas.metrics import faithfulness
    return _ragas_score("faithfulness", rows, faithfulness, "faithfulness")


def score_answer_relevance(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("answer_relevance", rows, _answer_relevance_proxy)
    from ragas.metrics import answer_relevancy
    return _ragas_score("answer_relevance", rows, answer_relevancy, "answer_relevancy")


def score_context_precision(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_precision", rows, _context_precision_proxy)
    from ragas.metrics import context_precision
    return _ragas_score("context_precision", rows, context_precision, "context_precision")


def score_context_recall(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_recall", rows, _context_recall_proxy)
    from ragas.metrics import context_recall
    return _ragas_score("context_recall", rows, context_recall, "context_recall")


# Convenience: all four at once.
ALL_METRICS = (
    score_faithfulness,
    score_answer_relevance,
    score_context_precision,
    score_context_recall,
)


def score_all(rows: list[dict]) -> dict[str, MetricResult]:
    results = [fn(rows) for fn in ALL_METRICS]
    return {r.name: r for r in results}


### RAGAS Metrics – Offline and Live Evaluation

This section defines the four RAG-specific evaluation metrics used in our project.

The `MetricResult` data class stores the metric name, evaluation mode, aggregate score and individual scores for each question.

The `offline_mode()` function determines whether the notebook should use offline proxy calculations or live RAGAS evaluation. Since `EVAL_OFFLINE=1` is set in our Setup section, this notebook uses the offline path.

The offline path uses token extraction and overlap calculations to estimate:

- **Faithfulness:** How much of the generated answer is supported by the retrieved context.
- **Answer relevance:** How closely the answer addresses the question.
- **Context precision:** What proportion of retrieved chunks are relevant to the reference answer.
- **Context recall:** How much of the reference answer's information appears in the retrieved context.

The four public scoring functions provide a common interface for both offline and live evaluation. `score_all()` runs all four metrics together and returns their results.

**CBSE adaptation:** The token extraction function retains short alphanumeric codes such as R1, R2 and R3 because these are meaningful terms in our CBSE questions.

**Important:** Offline scores are deterministic proxies based on token overlap. They are useful for understanding and testing the evaluation workflow, but they are not actual RAGAS scores or measurements of our live RAG application.

In [5]:

"""src/eval/harness.py — build the eval rows once, two ways.

Shared by eval/conftest.py (the pytest suite) and scripts/run_eval.py (the CLI),
so the offline synthetic pipeline and the live capstone pipeline are defined in
exactly one place.

Row shape (identical offline and live):
    {"id", "question", "answer", "contexts": [str,...], "ground_truth",
     "cache_hit", "latency_ms", "cost_usd", "notes"}
"""
from __future__ import annotations

from pathlib import Path

# GoldenEntry, load_golden_set, offline_mode are defined in the cells above

REPO_ROOT = Path.cwd()
GOLDEN_PATH = REPO_ROOT / "golden_set" / "capstone_v1.jsonl"

# KB version active at evaluation time.
# Offline pins the W10 value so every evaluation run is tagged consistently.
OFFLINE_KB_VERSION = "v2.w10"

# CBSE adaptation: an unrelated distractor for the retrieved contexts.
_DISTRACTOR = (
    "The weather forecast predicts cloudy skies and light rainfall "
    "in several regions during the coming week."
)


def _split_two(text: str) -> tuple[str, str]:
    """Split a reference answer into two context chunks."""
    words = text.split()
    mid = max(1, len(words) // 2)
    return " ".join(words[:mid]), " ".join(words[mid:])


def _synth_row(entry: GoldenEntry, index: int) -> dict:
    """Build one synthetic evaluation row for a golden-set entry.

    The reference answer is used as the synthetic answer.
    Its text is split into two relevant context chunks, with one unrelated
    distractor added to demonstrate context precision.
    """
    gt = entry.ground_truth

    # Divide the reference answer into two relevant chunks.
    c1, c2 = _split_two(gt)

    # Two relevant chunks and one unrelated distractor.
    contexts = [c1, c2, _DISTRACTOR]

    # Synthetic answer based on the reference answer.
    answer = gt

    # Retain the original cache and performance simulation.
    cache_hit = index < 13

    return {
        "id":           entry.id,
        "question":     entry.question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": gt,
        "cache_hit":    cache_hit,
        "latency_ms":   184 if cache_hit else 785,
        "cost_usd":     0.000041 if cache_hit else 0.000412,
        "notes":        entry.notes,
    }


def _live_rows(golden: list[GoldenEntry]) -> list[dict]:  # pragma: no cover - live only
    """Run the real capstone over the golden set. Needs the learner's src/rag."""
    try:
        from src.rag.qdrant_rag import ask_rag
        from src.rag.qdrant_store import QdrantStore
        from src.rag.retrieval import (
            build_bm25_index,
            hybrid_retrieve,
            load_chunks_from_qdrant,
        )
    except Exception as exc:
        raise RuntimeError(
            "Live eval needs your capstone code (src/rag/*) on the path and "
            "OPENAI_API_KEY + QDRANT_URL set. Run offline with EVAL_OFFLINE=1. "
            f"Import error: {exc}"
        ) from exc

    store = QdrantStore()
    chunks = load_chunks_from_qdrant(store)
    bm25 = build_bm25_index(chunks)
    rows: list[dict] = []

    for e in golden:
        res = ask_rag(e.question, store, k=3)

        # Full chunk text for RAGAS contexts comes from retrieval, not from
        # ask_rag's stripped citations.
        retrieved = hybrid_retrieve(store, bm25, chunks, e.question, top_k=3)

        rows.append({
            "id":           e.id,
            "question":     e.question,
            "answer":       res["answer"],
            "contexts":     [c.get("text", "") for c in retrieved],
            "ground_truth": e.ground_truth,
            "cache_hit":    res.get("cache_hit", False),
            "latency_ms":   res.get("latency_ms", 0),
            "cost_usd":     res.get("cost_usd", 0.0),
            "notes":        e.notes,
        })

    return rows


def active_kb_version() -> str:
    if offline_mode():
        return OFFLINE_KB_VERSION
    try:  # pragma: no cover - live only
        from src.rag.pipeline import KB_VERSION
        return KB_VERSION
    except Exception:  # pragma: no cover
        return "unknown"


def load_golden() -> list[GoldenEntry]:
    return load_golden_set(GOLDEN_PATH)


def build_rows(golden: list[GoldenEntry] | None = None) -> list[dict]:
    """Build the question, answer, contexts and ground-truth rows."""
    golden = golden or load_golden()

    if offline_mode():
        return [_synth_row(e, i) for i, e in enumerate(golden)]

    return _live_rows(golden)


### Evaluation Harness – Building the Evaluation Rows

The evaluation harness prepares the data required by the four RAG evaluation metrics and the KPI dashboard.

The `load_golden()` function reads the golden set, while `build_rows()` selects the offline or live evaluation path based on the evaluation mode.

In offline mode, `_synth_row()` creates one synthetic evaluation record for each golden-set question. It uses the reference answer as the simulated answer, splits the reference answer into two context chunks and adds one unrelated distractor.

The harness also assigns simulated cache status, latency and cost values. These additional fields will be used later when calculating the performance KPIs.

The `_live_rows()` function provides the alternative path for evaluating the actual RAG application. It retrieves real answers and context chunks from the application instead of creating synthetic records.

**CBSE adaptation:** The HR-specific distractor and deliberately seeded HR failures have been removed. The harness now uses the CBSE Class 9 golden set consistently.

**Important:** The offline rows are synthetic demonstration data. Their scores, latency and cost values do not represent actual performance of our RAG application.

### Build the Evaluation Rows and Identify the Knowledge Base Version

This section executes the evaluation harness defined in the previous section and prepares the data for the 8-KPI dashboard.

- **`rows = build_rows()`** calls the `build_rows()` function. Since our notebook is running in offline mode, it creates 20 synthetic evaluation records using the CBSE Class 9 golden set. Each record contains the question, simulated answer, retrieved context, reference answer, cache status, latency and cost.

- **`kb_version = active_kb_version()`** identifies the knowledge base version associated with this evaluation run. In offline mode, the harness uses the predefined version `v2.w10`.

- **`print()`** displays the number of evaluation records created and the knowledge base version assigned to this run.

**Expected output:**

`rows: 20 | kb_version: v2.w10`

**Why is this important?**

The evaluation rows are the input to the 8-KPI dashboard in the next section. The `kb_version` tag helps us identify which knowledge base version produced the evaluation results. This is important when comparing results across different versions.

**Note:** These are synthetic offline evaluation records, not results generated by our actual RAG application.

In [6]:
rows = build_rows()
kb_version = active_kb_version()
print('rows:', len(rows), '| kb_version:', kb_version)

rows: 20 | kb_version: v2.w10


### 8-KPI Dashboard – `src/eval/kpi.py`

This section defines the functions that assemble the eight KPIs from the evaluation records prepared earlier.

The purpose of this code is to bring together the four RAG evaluation metrics, task-success results, cost, latency and cache information into one dashboard.

**1. Constants and thresholds**

- `TOOL_SUCCESS_NA` – marks tool success as N/A because tool functionality is introduced in Week 13.
- `ESCALATION_NA` – marks escalation as N/A because human-in-the-loop functionality is introduced in Week 17.
- `_OFFLINE_GROUNDED_BAR` and `_OFFLINE_RELEVANT_BAR` – define the minimum faithfulness and answer-relevance scores required for an answer to count as successful in offline mode.

**2. Helper functions**

- `_percentile()` – calculates a requested percentile from a list of values. It is used to calculate latency p50 and p95.
- `_task_success_per_row()` – determines whether each answer is successful. In offline mode, both the faithfulness and answer-relevance scores must meet their thresholds. In live mode, the carried-forward Week 5 judge is used.
- `_rate()` – calculates the proportion of successful records.

**3. Main function: `compute_kpis()`**

This is the central function. It receives the evaluation rows and the active knowledge base version.

It first calls `score_all()` to obtain the four RAG evaluation metrics. It then calculates:

1. Task success rate
2. Grounded response rate
3. Hallucination rate (calculated as 1 minus faithfulness)
4. Retrieval hit rate
5. Tool success rate
6. Average cost per query
7. Latency p50 and p95
8. Escalation rate

It also calculates the additional Week 11 dimensions:

- `kb_version` – identifies the knowledge base version.
- `eval_mode` – indicates whether the scores are offline proxies or live evaluations.
- `cache_hit_rate` – proportion of queries served from cache.
- `cache_hit_accuracy` – task-success rate for cache-hit queries.
- `cache_miss_accuracy` – task-success rate for cache-miss queries.
- `n` – number of evaluation records.

**4. `reportable_count()`**

This function counts how many KPI categories contain numeric values and how many are marked N/A. Although latency is represented by two separate values (p50 and p95), it counts as one KPI category.

Therefore, the expected dashboard structure is **6 numeric KPIs + 2 N/A KPIs**.

**Important interpretation:** The code labels KPI 3 as `hallucination_rate` and calculates it as `1 - faithfulness`. As the lab guide clarifies, this is an unsupported-claim-density proxy, not the percentage of answers that contain hallucinations.

**Note:** This section only defines the dashboard functions. It does not calculate or display the KPI results yet. That happens in a later execution section.

In [7]:
"""src/eval/kpi.py — assemble the 8-KPI dashboard from scored rows.

This is the M2 deliverable's engine. It takes the scored eval rows and the
active kb_version and returns the full 8-KPI dashboard, plus the W11 extension
dimensions (cache-hit-rate and the cache-hit vs cache-miss accuracy split that
honours the W10 tombstone promise — D4).

The 8 KPIs (curriculum 'KPIs' sheet) and their W11 sources:
  1 Task success rate   — judge (live) / derived from grounded+relevant (offline)
  2 Grounded resp. rate  — RAGAS faithfulness
  3 Hallucination rate   — 1 - faithfulness (rough inverse)
  4 Retrieval hit rate   — RAGAS context precision
  5 Tool success rate    — N/A until W13 (no tools yet)
  6 Cost per query       — mean cost_usd (from results.db when present)
  7 Latency p50 / p95    — percentiles of latency_ms
  8 Escalation rate      — N/A until W17 (no HITL yet)
"""
from __future__ import annotations

# score_all, offline_mode are defined in the cells above

TOOL_SUCCESS_NA = "N/A (W13 — no tools yet)"
ESCALATION_NA = "N/A (W17 — no HITL yet)"

# KPI #1 task-success bars when derived offline (no judge available).
# Grounded bar is set just above the hallucinated row's faithfulness so a
# hallucination counts as a task failure; the relevant bar catches off-topic.
_OFFLINE_GROUNDED_BAR = 0.65
_OFFLINE_RELEVANT_BAR = 0.40


def _percentile(values: list[float], p: float) -> float:
    """Nearest-rank percentile. p in 0..100."""
    if not values:
        return 0.0
    s = sorted(values)
    k = max(0, min(len(s) - 1, round((p / 100.0) * (len(s) - 1))))
    return float(s[k])


def _task_success_per_row(rows, metrics) -> dict[str, bool]:
    """Per-row task success. Live: the W5 judge. Offline: grounded AND relevant."""
    faith = metrics["faithfulness"].per_row
    rel = metrics["answer_relevance"].per_row

    if offline_mode():
        return {
            r["id"]: (faith.get(r["id"], 0) >= _OFFLINE_GROUNDED_BAR
                      and rel.get(r["id"], 0) >= _OFFLINE_RELEVANT_BAR)
            for r in rows
        }

    # Live: grade with the carried-forward W5 judge.
    from .judge import judge_one_sync  # pragma: no cover - live only
    out: dict[str, bool] = {}
    for r in rows:  # pragma: no cover - live only
        score = judge_one_sync(r["question"], r["answer"], r["ground_truth"])
        out[r["id"]] = score.task_success
    return out


def _rate(flags: dict[str, bool]) -> float:
    return round(sum(flags.values()) / len(flags), 4) if flags else 0.0


def compute_kpis(rows: list[dict], kb_version: str) -> dict:
    """Return {'kpis', 'extras', 'metrics'} for the snapshot and the KPI test."""
    metrics = score_all(rows)                       # 4 RAGAS metrics, one pass
    faith = metrics["faithfulness"]
    cprec = metrics["context_precision"]

    success = _task_success_per_row(rows, metrics)

    costs = [r["cost_usd"] for r in rows]
    lats = [float(r["latency_ms"]) for r in rows]
    hits = [r for r in rows if r.get("cache_hit")]
    misses = [r for r in rows if not r.get("cache_hit")]

    kpis = {
        "1_task_success_rate":     _rate(success),
        "2_grounded_response_rate": faith.aggregate,
        "3_hallucination_rate":    round(1.0 - faith.aggregate, 4),
        "4_retrieval_hit_rate":    cprec.aggregate,
        "5_tool_success_rate":     TOOL_SUCCESS_NA,
        "6_cost_per_query_usd":    round(sum(costs) / len(costs), 6) if costs else 0.0,
        "7_latency_p50_ms":        _percentile(lats, 50),
        "7_latency_p95_ms":        _percentile(lats, 95),
        "8_escalation_rate":       ESCALATION_NA,
    }

    # W11 extension dimensions — kb_version tag + cache split (D4).
    hit_success = {r["id"]: success[r["id"]] for r in hits}
    miss_success = {r["id"]: success[r["id"]] for r in misses}
    extras = {
        "kb_version":         kb_version,
        "eval_mode":          faith.mode,
        "cache_hit_rate":     round(len(hits) / len(rows), 4) if rows else 0.0,
        "cache_hit_accuracy": _rate(hit_success),
        "cache_miss_accuracy": _rate(miss_success),
        "n":                  len(rows),
    }
    return {"kpis": kpis, "extras": extras, "metrics": metrics}


def reportable_count(kpis: dict) -> tuple[int, int]:
    """(numeric KPIs, N/A KPIs). Distinct KPI rows 1-8 (latency counts once)."""
    na = sum(1 for k, v in kpis.items() if isinstance(v, str) and v.startswith("N/A"))
    # KPIs 1-8, but latency is split across two keys -> count distinct rows.
    distinct = {k.split("_", 1)[0] for k in kpis}
    return len(distinct) - na, na


### What the 8-KPI Dashboard Implementation Defines

The previous code section defined the functions needed to calculate our dashboard. It did not execute those functions or produce KPI values.

This implementation provides the calculation logic for the next step:

- It combines the four RAG evaluation metrics using `score_all()`.
- It determines task success using faithfulness and answer relevance in offline mode.
- It calculates grounded response rate, hallucination proxy, retrieval hit rate, average cost and latency percentiles.
- It marks tool success and escalation as N/A because those capabilities are introduced in later weeks.
- It calculates the additional Week 11 cache-hit and cache-miss accuracy figures and attaches the `kb_version` tag.
- It uses `reportable_count()` to count the numeric KPI categories and N/A categories. Latency p50 and p95 count as one KPI.

**What happened when we executed this section?**

There was no output and no error. That is expected because this code only defines constants and functions. The dashboard has not yet been calculated.

**What comes next?**

The next section, titled **Compute the dashboard**, calls `compute_kpis(rows, kb_version)`. This is where the functions defined above are actually executed and the KPI values are displayed.

### Compute and Display the 8-KPI Dashboard

In the previous section, we defined the functions required to calculate the 8-KPI dashboard. Now we execute those functions using our 20 evaluation records.

**Step 1 – Calculate the dashboard**

`out = compute_kpis(rows, kb_version)`

This calls the main dashboard function with two inputs:

- `rows` – our 20 evaluation records.
- `kb_version` – the knowledge base version, `v2.w10`.

The function calculates the eight KPIs and the additional Week 11 information. It returns these results in a dictionary called `out`.

**Step 2 – Separate the results**

`kpis, extras = out['kpis'], out['extras']`

This extracts two parts of the returned dictionary:

- `kpis` contains the eight KPI results.
- `extras` contains the Week 11 extensions, such as cache-hit rate, cache-hit accuracy, cache-miss accuracy and knowledge base version.

**Step 3 – Display the eight KPIs**

The `for` loop goes through each KPI name and its calculated value and prints them.

**Step 4 – Display the Week 11 extensions**

The next `for` loop displays the additional information associated with this evaluation run.

**Step 5 – Count numeric and N/A KPIs**

`reportable_count(kpis)` counts the numeric KPI categories and the N/A categories.

The final print statement displays this count.

**What we expect to observe:**

- Eight KPI categories.
- Six numeric KPI categories.
- Two N/A categories: tool success and escalation.
- Additional Week 11 information, including the `v2.w10` knowledge base tag and the cache-hit versus cache-miss accuracy split.

This is where the dashboard calculation defined in the previous section is actually executed.

In [8]:
out = compute_kpis(rows, kb_version)
kpis, extras = out['kpis'], out['extras']

print('=== 8-KPI dashboard ===')
for k, v in kpis.items():
    print(f'  {k:28s} {v}')

print('\n=== W11 extensions ===')
for k, v in extras.items():
    print(f'  {k:20s} {v}')

numeric, na = reportable_count(kpis)
print(f'\nreportable: {numeric} numeric + {na} N/A')

=== 8-KPI dashboard ===
  1_task_success_rate          1.0
  2_grounded_response_rate     1.0
  3_hallucination_rate         0.0
  4_retrieval_hit_rate         0.6667
  5_tool_success_rate          N/A (W13 — no tools yet)
  6_cost_per_query_usd         0.000171
  7_latency_p50_ms             184.0
  7_latency_p95_ms             785.0
  8_escalation_rate            N/A (W17 — no HITL yet)

=== W11 extensions ===
  kb_version           v2.w10
  eval_mode            offline-proxy
  cache_hit_rate       0.65
  cache_hit_accuracy   1.0
  cache_miss_accuracy  1.0
  n                    20

reportable: 6 numeric + 2 N/A


### Interpreting the 8-KPI Dashboard Results

The dashboard has been calculated successfully using our 20 synthetic CBSE evaluation records.

**Main observations:**

1. **Task success rate – 1.0:** All 20 answers passed the offline task-success conditions, which require both faithfulness and answer relevance to meet their thresholds.

2. **Grounded response rate – 1.0:** All simulated answers are fully supported by their corresponding contexts.

3. **Hallucination rate – 0.0:** The unsupported-claim-density proxy is zero because the simulated answers contain no additional unsupported claims.

4. **Retrieval hit rate – 0.6667:** Context precision is approximately 66.67%. Each record contains two relevant context chunks and one unrelated distractor.

5. **Tool success and escalation – N/A:** These capabilities are introduced in Weeks 13 and 17 respectively.

6. **Cost per query – 0.000171 USD:** This is the average simulated cost across the 20 records.

7. **Latency – p50: 184 ms; p95: 785 ms:** These values represent the median and 95th-percentile simulated response times.

**Week 11 extensions:**

- The evaluation is tagged with knowledge base version `v2.w10`.
- The evaluation mode is `offline-proxy`.
- Cache-hit rate is 65%.
- Both cache-hit and cache-miss task-success rates are 100%.
- A total of 20 records were evaluated.

**Final checkpoint:** The dashboard reports 6 numeric KPI categories and 2 N/A categories, as expected.

**Important limitation:** These results are based on synthetic offline data. They demonstrate the KPI calculation and reporting workflow, not the actual performance of our live RAG application.

The next stage of the notebook is to persist the evaluation results in `results.db`, so that results can be retained and grouped by knowledge base version.

## 2 · Persist per-query cost/latency — `scripts/init_results_db.py`

At the end of W10, `results.db` held only the semantic-cache tables. W11 adds one small `eval_results` table logging per-query cost/latency/cache-hit **tagged by `kb_version`**, so the snapshot is reproducible without a live re-run — and so you never aggregate across two KB versions by accident.

In the previous section, we calculated the 8-KPI dashboard. Now we prepare the database to store the per-query evaluation information so that results can be retained and examined later.

At the end of Week 10, our `results.db` database contains only the semantic-cache tables:

- `query_cache`
- `query_cache_meta`

Week 11 introduces a new table called `eval_results` to store evaluation-related information.

**1. Database connection and default path**

- `sqlite3` provides Python's interface for working with SQLite databases.
- `DEFAULT_DB = "./results.db"` specifies the default database file.
- `Path` and `sys` are imported for filesystem and command-line operations.

**2. Database table structure – `SCHEMA`**

The SQL statement creates a table named `eval_results`, if it does not already exist.

The table stores:

- `run_label` – identifies an evaluation run.
- `kb_version` – identifies the knowledge base version.
- `golden_id` – identifies the golden-set question.
- `cache_hit` – records whether the query was served from cache.
- `latency_ms` – records query latency in milliseconds.
- `cost_usd` – records the query cost in US dollars.
- `created_at` – records when the database entry was created.

The combination of `run_label` and `golden_id` forms the primary key, preventing duplicate entries for the same question within the same run.

**3. `init_db()` function**

This function connects to the SQLite database, executes the table-creation statement, commits the change and closes the connection.

**4. `write_rows()` function**

This function saves evaluation records into the `eval_results` table.

It:
- Ensures that the table exists by calling `init_db()`.
- Opens a database connection.
- Uses `executemany()` to insert multiple records.
- Uses `INSERT OR REPLACE` so an existing record with the same primary key is replaced.
- Commits the changes and returns the number of rows written.
- Closes the connection.

**Why are we doing this?**

Saving the per-query cost, latency and cache information makes the evaluation results reproducible. We can later group the records by `kb_version` and compare evaluation runs without having to repeat the original live evaluation.

**Important:** This section only defines the database schema and the functions. The table creation and data insertion will be performed in the following code section.

In [9]:
"""scripts/init_results_db.py — add an eval-results table to results.db.

Reality check from the plan: at end of W10, `results.db` holds only the
semantic-cache tables (`query_cache`, `query_cache_meta`). The curriculum
says cost/latency come "from results.db", so W11 adds one small table that
logs per-query cost/latency/cache-hit for an eval run. test_kpi.py reads from
it when present (and falls back to the in-memory rows otherwise), so the
snapshot is reproducible without a live re-run.

Usage:
    python3 scripts/init_results_db.py            # create the table
    # then run_eval.py --persist writes rows into it
"""
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

DEFAULT_DB = "./results.db"

SCHEMA = """
CREATE TABLE IF NOT EXISTS eval_results (
    run_label   TEXT    NOT NULL,
    kb_version  TEXT    NOT NULL,
    golden_id   TEXT    NOT NULL,
    cache_hit   INTEGER NOT NULL,
    latency_ms  INTEGER NOT NULL,
    cost_usd    REAL    NOT NULL,
    created_at  TEXT    DEFAULT (datetime('now')),
    PRIMARY KEY (run_label, golden_id)
);
"""


def init_db(db_path: str = DEFAULT_DB) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(SCHEMA)
        conn.commit()
    finally:
        conn.close()


def write_rows(rows: list[dict], run_label: str, kb_version: str,
               db_path: str = DEFAULT_DB) -> int:
    """Persist per-query cost/latency for an eval run. Returns rows written."""
    init_db(db_path)
    conn = sqlite3.connect(db_path)
    try:
        conn.executemany(
            "INSERT OR REPLACE INTO eval_results "
            "(run_label, kb_version, golden_id, cache_hit, latency_ms, cost_usd) "
            "VALUES (?,?,?,?,?,?)",
            [(run_label, kb_version, r["id"], int(bool(r.get("cache_hit"))),
              int(r.get("latency_ms", 0)), float(r.get("cost_usd", 0.0)))
             for r in rows],
        )
        conn.commit()
        return len(rows)
    finally:
        conn.close()

# (The module's command-line entry point is intentionally omitted in the notebook.
#  Call init_db(...) and write_rows(...) directly instead, as the next cell does.)


### Understanding the Database Implementation

We have now defined the database structure and the functions required to persist evaluation results.

**What this code has established:**

- The `eval_results` table structure is ready to be created.
- `init_db()` provides the functionality to create the table.
- `write_rows()` provides the functionality to insert or replace evaluation records.
- Each record is associated with a run label and a knowledge base version.

**Has the database been changed yet?**

Not by executing this definition cell alone. Defining `init_db()` and `write_rows()` does not automatically call them. The database table is created only when `init_db()` is called, and evaluation records are saved only when `write_rows()` is called.

**What comes next?**

The following section will call these functions to create the `eval_results` table, write our 20 evaluation records into it, and read the stored information back. We will then examine how the records are grouped by `kb_version`.

This completes the database implementation section.

### Save Evaluation Results in SQLite and Verify the Stored Records

**Write this run's rows, then read them back** (grouped by `kb_version`):


In the previous section, we defined two functions: `init_db()` and `write_rows()`. This section executes those functions to save our evaluation results in the SQLite database.

**1. Create the database table**

- `import sqlite3` imports Python's SQLite database module.
- `init_db('results.db')` creates the `eval_results` table in `results.db`, if it does not already exist.

**2. Write the evaluation records**

- `write_rows()` saves the evaluation records contained in `rows`.
- `run_label='wk11-baseline'` gives this evaluation run a name.
- `kb_version=kb_version` associates the records with the knowledge base version identified earlier (`v2.w10`).
- `db_path='results.db'` specifies the database file.
- The returned number of inserted or replaced records is stored in `n`.

**3. Display the number of records saved**

The `print()` statement displays how many evaluation records were persisted.

**4. Read the stored information**

The code opens a connection to `results.db` and executes an SQL query.

The query uses:

- `SELECT` to retrieve the required columns.
- `COUNT(*)` to count the records.
- `AVG(cost_usd)` to calculate the average cost per query.
- `AVG(latency_ms)` to calculate the average latency.
- `GROUP BY kb_version` to group the results by knowledge base version.

Finally, the connection is closed and the query results are displayed.

**Why is this important?**

The database now provides a persistent record of evaluation costs and latency. We can use the `kb_version` field to distinguish results from different knowledge base versions and compare them separately.

In [10]:
import sqlite3
init_db('results.db')
n = write_rows(rows, run_label='wk11-baseline', kb_version=kb_version, db_path='results.db')
print('persisted', n, 'rows')

conn = sqlite3.connect('results.db')
q = conn.execute('SELECT kb_version, COUNT(*), ROUND(AVG(cost_usd),6), ROUND(AVG(latency_ms),1) '
                 'FROM eval_results GROUP BY kb_version').fetchall()
conn.close()
print('kb_version | rows | avg cost | avg latency')
for row in q:
    print('  ', row)

persisted 20 rows
kb_version | rows | avg cost | avg latency
   ('v2.w10', 20, 0.000171, 394.4)


### Understanding the Persisted Evaluation Results

The database operation completed successfully.

**1. Records persisted**

The output `persisted 20 rows` confirms that all 20 evaluation records were saved in the `eval_results` table.

**2. Results grouped by knowledge base version**

The SQL query returned:

| Field | Result | Meaning |
|---|---:|---|
| Knowledge base version | `v2.w10` | Version associated with this evaluation run |
| Number of records | 20 | All 20 questions were stored |
| Average cost | $0.000171 | Average simulated cost per query |
| Average latency | 394.4 ms | Average simulated response time |

**3. Understanding the average latency**

Our synthetic records contain two latency values:

- Cache hits: 13 records × 184 ms
- Cache misses: 7 records × 785 ms

The average is:

\[
\frac{(13 \times 184)+(7 \times 785)}{20}
=394.35 \text{ ms}
\]

Rounded to one decimal place, this gives **394.4 ms**.

**Why is this important?**

We have now moved beyond calculating KPIs in memory. The evaluation records are stored persistently in `results.db`, along with their knowledge base version.

This allows us to retrieve and compare evaluation runs later. When comparing different knowledge base versions, we should keep their results separate rather than combining them into one average.

**Important:** These are still synthetic offline evaluation records. Persisting them in the database does not make them measurements from our live RAG application.

The next section moves to the baseline evaluation report.

## 3 · The baseline report — `scripts/run_eval.py`
### Formatting the Evaluation Results

In the previous sections, we calculated the 8-KPI dashboard and saved the per-query evaluation information in `results.db`.

This section defines `_render_report()`, a function that converts the evaluation results into a readable Markdown report.

The function receives four inputs:

- `rows` – the evaluation records for the golden-set questions.
- `kpis` – the eight KPI values calculated by `compute_kpis()`.
- `extras` – additional information such as evaluation mode, knowledge base version and cache statistics.
- `metrics` – the four RAG evaluation metric results.

**1. Identify the evaluation mode**

The function reads `eval_mode` from `extras`.

If the mode is `offline-proxy`, it adds a warning banner explaining that the results are deterministic token-overlap proxies, not actual RAGAS scores.

**2. Prepare the report heading and summary**

The `lines` list begins building the report. It includes:

- Report title
- Evaluation mode
- Knowledge base version
- Number of golden-set entries

**3. Display the four RAG metrics**

The function creates a table containing:

- Faithfulness
- Answer relevance
- Context precision
- Context recall

The aggregate score for each metric is displayed to three decimal places.

**4. Display the 8-KPI dashboard**

The `rowmap` list connects each KPI number and label to its corresponding value in the `kpis` dictionary.

The function then creates a Markdown table showing all eight KPIs. Latency p50 and p95 are displayed together in one row.

**5. Display Week 11 extensions**

This section adds the cache-hit rate, cache-hit accuracy, cache-miss accuracy and knowledge base version.

**6. Display the lowest-faithfulness entries**

The function sorts the evaluation records by their faithfulness scores, from lowest to highest, and displays the first six records.

This helps identify which questions have the weakest faithfulness scores and may need further investigation.

**7. Return the report**

Finally, `return "".join(lines)` combines all the prepared lines into one Markdown string.

**Important:** This cell only defines the report-rendering function. It does not generate or display the report yet. The next execution section must call `_render_report()` and display or save the returned text.

The *produce-real-numbers-on-demand* entry point renders a markdown baseline report. Offline output is **banner-marked as proxy numbers** — never paste those into the M2 dashboard; run live for the numbers you defend. Here is its report renderer verbatim, called directly (the notebook replaces the module's `argparse` CLI wrapper).

In [11]:
def _render_report(rows, kpis, extras, metrics) -> str:
    mode = extras["eval_mode"]
    banner = ""
    if mode == "offline-proxy":
        banner = (
            "> ⚠️ **MODE: offline-proxy.** These are deterministic token-overlap "
            "proxies for CI, **not** RAGAS numbers. Run live (`python3 "
            "scripts/run_eval.py`) for the numbers you put in the snapshot.\n\n"
        )
    lines = [
        "# wk11 RAGAS baseline (auto-generated)\n",
        banner,
        f"- **eval mode:** `{mode}`\n",
        f"- **kb_version:** `{extras['kb_version']}`\n",
        f"- **golden set:** {extras['n']} entries\n",
        "\n## RAG metrics (RAGAS)\n",
        "| Metric | Aggregate |\n|---|---|\n",
    ]
    for name in ("faithfulness", "answer_relevance",
                 "context_precision", "context_recall"):
        lines.append(f"| {name} | {metrics[name].aggregate:.3f} |\n")

    lines.append("\n## 8-KPI dashboard\n")
    lines.append("| # | KPI | Value |\n|---|---|---|\n")
    rowmap = [
        ("1", "Task success rate",     kpis["1_task_success_rate"]),
        ("2", "Grounded response rate", kpis["2_grounded_response_rate"]),
        ("3", "Hallucination rate",    kpis["3_hallucination_rate"]),
        ("4", "Retrieval hit rate",    kpis["4_retrieval_hit_rate"]),
        ("5", "Tool success rate",     kpis["5_tool_success_rate"]),
        ("6", "Cost per query (USD)",  kpis["6_cost_per_query_usd"]),
        ("7", "Latency p50 / p95 (ms)",
         f"{kpis['7_latency_p50_ms']:.0f} / {kpis['7_latency_p95_ms']:.0f}"),
        ("8", "Escalation rate",       kpis["8_escalation_rate"]),
    ]
    for num, label, val in rowmap:
        lines.append(f"| {num} | {label} | {val} |\n")

    lines.append("\n## W11 extensions (W10 tombstone payoff)\n")
    lines.append(
        f"- cache hit rate: {extras['cache_hit_rate']}\n"
        f"- cache-hit accuracy: {extras['cache_hit_accuracy']} · "
        f"cache-miss accuracy: {extras['cache_miss_accuracy']}\n"
        f"- results tagged `kb_version={extras['kb_version']}`\n"
    )

    lines.append("\n## Per-entry (lowest faithfulness first)\n")
    faith = metrics["faithfulness"].per_row
    lines.append("| id | faithfulness | notes |\n|---|---|---|\n")
    order = sorted(rows, key=lambda r: faith.get(r["id"], 0.0))
    for r in order[:6]:
        note = (r.get("notes", "") or "").split(".")[0][:48]
        lines.append(f"| {r['id']} | {faith[r['id']]:.3f} | {note} |\n")
    return "".join(lines)


### Understanding the Baseline Report Renderer

The report-rendering function has been defined successfully.

**What has happened?**

The function `_render_report()` is now available to format our evaluation results into a Markdown report.

It is designed to bring together:

- The evaluation mode and knowledge base version.
- The four RAG evaluation metric scores.
- The eight KPI dashboard values.
- The Week 11 cache-related statistics.
- The six entries with the lowest faithfulness scores.

**Why was there no output?**

This cell only defines a function. It does not call `_render_report()`, so no report is generated or displayed yet.

**What comes next?**

The next execution section will call the function using our existing evaluation results and display or save the generated baseline report.

**Important:** Because our evaluation mode is `offline-proxy`, the report will explicitly identify the results as deterministic offline proxies, not actual RAGAS measurements.

### Generate and Display the Baseline Evaluation Report

**Render and display the report:**


In the previous section, we defined `_render_report()`, which contains the logic for formatting our evaluation results into a Markdown report.

This section executes that function and displays the generated report.

**1. Import the display functions**

`from IPython.display import Markdown, display`

- `Markdown` allows us to render text using Markdown formatting inside the notebook.
- `display` presents the formatted content in the notebook output area.

**2. Retrieve the evaluation metrics**

`metrics = out['metrics']`

The variable `out` contains the results returned by `compute_kpis()`.

This statement retrieves the four RAG evaluation metric results from it.

**3. Generate the report**

`report = _render_report(rows, kpis, extras, metrics)`

This calls the report-rendering function with:

- `rows` – our 20 evaluation records.
- `kpis` – the calculated dashboard values.
- `extras` – evaluation mode, knowledge base version and cache statistics.
- `metrics` – the four RAG evaluation metric results.

The generated Markdown report is stored in the variable `report`.

**4. Display the report**

`display(Markdown(report))`

This renders the report as a formatted Markdown document in the notebook output area, including headings and tables.

**Why is this important?**

We now bring together the evaluation metrics, KPI dashboard and cache statistics into one readable baseline report. This provides a consolidated view of the current evaluation run.

In [12]:
from IPython.display import Markdown, display
metrics = out['metrics']
report = _render_report(rows, kpis, extras, metrics)
display(Markdown(report))

# wk11 RAGAS baseline (auto-generated)
> ⚠️ **MODE: offline-proxy.** These are deterministic token-overlap proxies for CI, **not** RAGAS numbers. Run live (`python3 scripts/run_eval.py`) for the numbers you put in the snapshot.

- **eval mode:** `offline-proxy`
- **kb_version:** `v2.w10`
- **golden set:** 20 entries

## RAG metrics (RAGAS)
| Metric | Aggregate |
|---|---|
| faithfulness | 1.000 |
| answer_relevance | 0.792 |
| context_precision | 0.667 |
| context_recall | 1.000 |

## 8-KPI dashboard
| # | KPI | Value |
|---|---|---|
| 1 | Task success rate | 1.0 |
| 2 | Grounded response rate | 1.0 |
| 3 | Hallucination rate | 0.0 |
| 4 | Retrieval hit rate | 0.6667 |
| 5 | Tool success rate | N/A (W13 — no tools yet) |
| 6 | Cost per query (USD) | 0.000171 |
| 7 | Latency p50 / p95 (ms) | 184 / 785 |
| 8 | Escalation rate | N/A (W17 — no HITL yet) |

## W11 extensions (W10 tombstone payoff)
- cache hit rate: 0.65
- cache-hit accuracy: 1.0 · cache-miss accuracy: 1.0
- results tagged `kb_version=v2.w10`

## Per-entry (lowest faithfulness first)
| id | faithfulness | notes |
|---|---|---|
| g001 | 1.000 | happy-path; source: CBSE Circular Acad-33/2026 a |
| g002 | 1.000 | happy-path; source: CBSE Circular Acad-33/2026 a |
| g003 | 1.000 | happy-path; source: CBSE Circular Acad-33/2026;  |
| g004 | 1.000 | happy-path; source: Secondary Curriculum Part I; |
| g005 | 1.000 | happy-path; source: Secondary Curriculum Part I |
| g006 | 1.000 | happy-path; source: English Language and Literat |


### Interpreting the Week 11 Baseline Evaluation Report

The baseline report has been generated successfully using our 20 CBSE Class 9 evaluation records.

**1. Evaluation information**

- Evaluation mode: `offline-proxy`
- Knowledge base version: `v2.w10`
- Golden-set size: 20 questions

The report explicitly warns that these are deterministic token-overlap proxy scores, not actual RAGAS evaluation results.

**2. Four RAG evaluation metrics**

| Metric | Score | Interpretation |
|---|---:|---|
| Faithfulness | 1.000 | All simulated answers are supported by their retrieved contexts. |
| Answer relevance | 0.792 | The simulated answers show approximately 79.2% relevance to the questions. |
| Context precision | 0.667 | Approximately 66.7% of the retrieved context chunks are relevant. |
| Context recall | 1.000 | All reference-answer information is represented in the retrieved contexts according to the offline proxy. |

**3. Eight-KPI dashboard**

- Task success rate: 1.0
- Grounded response rate: 1.0
- Hallucination rate: 0.0
- Retrieval hit rate: 0.6667
- Tool success rate: N/A – introduced in Week 13.
- Cost per query: $0.000171
- Latency p50/p95: 184 ms / 785 ms
- Escalation rate: N/A – introduced in Week 17.

**4. Week 11 extensions**

- Cache-hit rate: 65%.
- Cache-hit accuracy: 100%.
- Cache-miss accuracy: 100%.
- Results are tagged with knowledge base version `v2.w10`.

**5. Per-entry faithfulness**

The report sorts the evaluation records from lowest to highest faithfulness and displays the first six entries.

In our current output, g001 through g006 all have a faithfulness score of 1.000. This means there are no lower-scoring entries among the displayed six.

**Overall interpretation**

We have successfully generated a consolidated baseline report containing the four RAG metrics, eight KPIs, cache statistics and per-question faithfulness information.

The report also preserves an important distinction: our results demonstrate the evaluation and reporting workflow using synthetic offline data. They do not establish the actual performance of our live CBSE RAG application.

**What comes next?**

The next notebook section is **4 · Thresholds from a baseline**. There, we will use the baseline results to establish KPI thresholds and understand how evaluation results can be assessed against those thresholds.

## 4 · Thresholds from a baseline

A threshold on the floor tests nothing. Read your baseline aggregate, set the bar **just below it** — a real regression trips it; run-to-run judge noise does not — and record the number *and the reason* in the ADR. This is exactly what the pytest suite asserts on every commit; the same check, inline:

### Set Evaluation Thresholds and Compare Against the Baseline

In the previous section, we generated the baseline evaluation report using our 20 CBSE Class 9 questions.

This section defines minimum acceptable scores, called **thresholds**, for the four RAG evaluation metrics.

**1. Define the thresholds**

The `THRESHOLDS` dictionary specifies the minimum acceptable score for each metric:

| Metric | Threshold |
|---|---:|
| Faithfulness | 0.85 |
| Answer relevance | 0.85 |
| Context precision | 0.75 |
| Context recall | 0.80 |

These values are taken from the notebook's stated baseline-set bars.

**2. Compare aggregate scores against thresholds**

The `for` loop goes through each metric and its threshold.

For each metric:

- `metrics[name].aggregate` retrieves its aggregate evaluation score.
- `bar` represents the minimum acceptable score.
- `agg >= bar` checks whether the aggregate score meets or exceeds the threshold.

**3. Display the comparison**

The `print()` statements display a comparison table containing:

- Metric name
- Actual aggregate score
- Threshold value
- Whether the metric passes (`True`) or fails (`False`)

**Why are we doing this?**

A metric score by itself tells us how the system performed. A threshold gives us a reference point for deciding whether that performance is acceptable.

**Important:** These thresholds are intended to be set just below the corresponding live baseline aggregates. Our current evaluation uses offline proxies, so some metrics may not pass. The notebook explicitly states that this is expected. The purpose here is to practise the threshold-comparison workflow.

In [13]:
# baseline-set bars (see the deck / ADR): set just below the live baseline aggregate
THRESHOLDS = {'faithfulness': 0.85, 'answer_relevance': 0.85,
              'context_precision': 0.75, 'context_recall': 0.80}

print('metric              aggregate   bar     pass?')
for name, bar in THRESHOLDS.items():
    agg = metrics[name].aggregate
    print(f'  {name:18s} {agg:.3f}      {bar:.2f}    {agg >= bar}')

# (offline proxies won't all clear live-set bars — that's expected; the point is the
#  wiring. Run live, then set each bar just below the real aggregate.)

metric              aggregate   bar     pass?
  faithfulness       1.000      0.85    True
  answer_relevance   0.792      0.85    False
  context_precision  0.667      0.75    False
  context_recall     1.000      0.80    True


### Interpreting the Baseline Threshold Comparison

The threshold comparison executed successfully for all four RAG evaluation metrics.

**1. Results**

| Metric | Aggregate | Threshold | Result |
|---|---:|---:|---|
| Faithfulness | 1.000 | 0.85 | Pass |
| Answer relevance | 0.792 | 0.85 | Fail |
| Context precision | 0.667 | 0.75 | Fail |
| Context recall | 1.000 | 0.80 | Pass |

**2. What do these results tell us?**

- **Faithfulness passes:** The score of 1.000 is above the minimum threshold of 0.85.
- **Answer relevance fails:** The score of 0.792 is below the threshold of 0.85.
- **Context precision fails:** The score of 0.667 is below the threshold of 0.75.
- **Context recall passes:** The score of 1.000 is above the threshold of 0.80.

Therefore, two metrics pass and two metrics fail.

**3. Why do two metrics fail?**

Our current evaluation uses synthetic offline data and deterministic token-overlap proxies. The answer relevance and context precision scores do not reach the predefined threshold values.

This is expected. The notebook explains that these thresholds are intended to sit just below the actual live baseline aggregates. They should be adjusted after running a live evaluation, not simply lowered to make the offline results pass.

**4. What have we learned?**

Thresholds provide a way to assess evaluation results against predefined minimum requirements.

Instead of looking only at metric scores, we can identify which evaluation dimensions meet the required standard and which need attention.

**Important:** This comparison demonstrates the threshold-checking workflow. It does not establish whether our live CBSE RAG application meets these performance requirements.

This completes the threshold-comparison exercise.

## Where this graduates to

These notebooks are the place to **see and explain** the eval — run a cell, read a number, talk to it. The one thing a notebook can't do is gate a deploy or run on every commit, so in the package this same code lives as a **pytest suite** (`eval/test_*.py`) that runs offline in CI and produces the live numbers on demand. Notebook to understand it; the suite to enforce it. Next week (W12) is the eval regression runbook and DR #2, where you present these numbers and defend your thresholds.